In [107]:
####################################
#ENVIRONMENT SETUP

In [108]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle
import glob

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime, timedelta

#DownloadERA5Data functions
import cdsapi

In [109]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [110]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "ERA5_Data")
dataType = "ERA5Comparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Data Directory:           /glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/ERA5_Data



In [111]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [274]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

spinup_hours = "24"
# spinup_hours = "12"

# RunType = ("TRACER","MOIST","NSSL",spinup_hours)
RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

Found 289/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/model_run_spinup24hrs/history_cartesian/history.2022-06-30_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/model_run_spinup24hrs/diag_cartesian/diag.2022-06-30_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           MOIST
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-30 to 2022-07-03
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:289
 # Diag Files:   289
 # Time Steps:   289
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/model_run_spinup24hrs
 Static File:    TRACER_regional52

In [275]:
####################################
#DOWNLOAD

In [276]:
variables = ERA5DataLoading_Class.GetVariableNames_Surface()
date = ERA5DataLoading_Class.GetERA5Date(ModelData.SimulationTime)
area = ERA5DataLoading_Class.GetERA5Area(ModelData.latitude,ModelData.longitude)
ERA5FilePath = ERA5DataLoading_Class.GetERA5FilePath(DirectoryManager,ModelData)

In [ ]:
ERA5DataLoading_Class.DownloadERA5_Surface(variables, date, area, ERA5FilePath)

In [290]:
# ============================================================
# ERA5DataLoading_Class 
# ============================================================

#Libraries
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr

import cdsapi 

class ERA5DataLoading_Class:

    @staticmethod
    def GetERA5FileName_PressureLevels(ERA5FilePath, variable):
        fileName = f"ERA5Surface_{variable}.nc"
        ERA5FileName = os.path.join(ERA5FilePath, fileName)
        return ERA5FileName

    @staticmethod
    def GetERA5FileName_Surface(ERA5FilePath, variable):
        fileName = f"ERA5Surface_{variable}.nc"
        ERA5FileName = os.path.join(ERA5FilePath, fileName)
        return ERA5FileName
        
    @staticmethod
    def DownloadERA5_PressureLevels(variables, date, area, ERA5FilePath):
        """
        # DOWNLOADING ERA5 (Pressure Levels)
        # Code Inspired from "Download_ERA5_with_python" by github.com/joaohenry23 at https://github.com/joaohenry23/Download_ERA5_with_python
        
        #PRELIMINARY STEPS
        #(1) go to https://cds.climate.copernicus.eu/how-to-api
        #(2) make file in main user directory called .cdsapirc
        #(3) copy the following into file: 
        #    url: https://cds.climate.copernicus.eu/api
        #    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
        #(4) pip install "cdsapi>=0.7.4"
        """
        c = cdsapi.Client()
        for variable in tqdm(variables, desc="Downloading ERA5 variables"):
            print(f"Downloading {variable}", "\n")
            
            ERA5FileName = ERA5DataLoading_Class.GetERA5FileName_PressureLevels(ERA5FilePath, variable)
            print(f'Saving file to {ERA5FileName}')
            
            c.retrieve(
                "reanalysis-era5-pressure-levels",
                {
                    "product_type": "reanalysis",
                    "format": "netcdf",
                    "variable": variable,
                    # "pressure_level": ['100', '250', '500', '750', '1000'], #LOW-RES
                    "pressure_level": [
                        '10', '20', '30', '50', '70', 
                        '100', '125', '150', '175', '200', '225',
                        '250', '300', '350', '400', '450', '500',
                        '550', '600', '650', '700', '750', '775',
                        '800', '825', '850', '875', '900', '925',
                        '950', '975', '1000',
                    ],
    
                    "date": date,
                    "time": [f"{h:02d}:00" for h in range(24)],
                    "area": area,
                    "grid": [0.25, 0.25],
                },
                ERA5FileName
            )

    @staticmethod
    def DownloadERA5_Surface(variables, date, area, ERA5FilePath):
        """
        #PRELIMINARY STEPS
        #(1) go to https://cds.climate.copernicus.eu/how-to-api
        #(2) make file in main user directory called .cdsapirc
        #(3) copy the following into file: 
        #    url: https://cds.climate.copernicus.eu/api
        #    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
        #(4) pip install "cdsapi>=0.7.4"
        """
        c = cdsapi.Client()
        for variable in tqdm(variables, desc="Downloading ERA5 surface variables"):
            print(f"Downloading {variable}", "\n")
    
            ERA5FileName = ERA5DataLoading_Class.GetERA5FileName_Surface(ERA5FilePath, variable)
            print(f'Saving file to {ERA5FileName}')
            
            c.retrieve(
                "reanalysis-era5-single-levels",
                {
                    "product_type": "reanalysis",
                    "format": "netcdf",
                    "variable": variable,
                    "date": date,  # e.g. "2022-06-08/2022-06-10"
                    "time": [f"{h:02d}:00" for h in range(24)],  # every hour
                    "area": area,  # [north, west, south, east]
                    "grid": [0.25, 0.25],
                },
                ERA5FileName
            )
        
    @staticmethod
    def GetVariableNames_PressureLevels():
        variables = [
            "u_component_of_wind",
            "v_component_of_wind",
            "vertical_velocity",
            "divergence",
            "vorticity",
            "temperature",
            "specific_humidity",
            "specific_cloud_liquid_water_content",
            "specific_cloud_ice_water_content",
            "specific_rain_water_content",
            "relative_humidity",
            "geopotential",
        ]
        return variables

    @staticmethod
    def GetVariableNames_Surface():
        """
        #for list of variables:
        # https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download
        """
        variables = ["msl",
                     "total_precipitation",
                     "2m_temperature",
                     "surface_sensible_heat_flux",
                     "surface_latent_heat_flux"]
        return variables
    
    @staticmethod
    def GetERA5FolderName(timeString):
        dt = datetime.strptime(timeString, "%Y-%m-%d_%H.%M.%S")
        year = dt.year
        month = f"{dt.month:02d}"
        folderName = f"{year}{month}"
        return folderName

    @staticmethod
    def GetERA5Date(dateTuple):
        startDate, endDate = dateTuple
    
        # convert endDate string → datetime
        end = datetime.strptime(endDate, "%Y-%m-%d")
    
        # subtract one day because ERA5 uses inclusive ranges
        realEnd = (end - timedelta(days=1)).strftime("%Y-%m-%d")
    
        # return ERA5 format
        return f"{startDate}/{realEnd}"

    @staticmethod
    def GetERA5Area(lat, lon):
        """
        lat, lon can be 1D or 2D arrays from ModelData_NSSL.
        Returns area in ERA5 format: [N, W, S, E].
        """
        lon360 = lon+360
        
        north = float(np.max(lat))
        south = float(np.min(lat))
        east  = float(np.max(lon360))
        west  = float(np.min(lon360))
    
        return [north, west, south, east]

    @staticmethod
    def GetERA5FilePath(DirectoryManager,ModelData):
        ERA5FilePath = os.path.join(DirectoryManager.dataDirectory,
                                      "ERA5_Data",
                                      ModelData.region,
                                      ModelData.case)
        os.makedirs(ERA5FilePath, exist_ok=True)
        return ERA5FilePath
        
    @staticmethod
    def SubsetERA5(ERA5Data, ModelData):
        latModel = ModelData.latitude
        lonModel = ModelData.longitude
    
        # Convert model longitudes from -180:180 → 0:360 to match ERA5
        lonModel_360 = (lonModel + 360) % 360
    
        # Determine model bounding box
        lat_min, lat_max = float(latModel.min()), float(latModel.max())
        lon_min, lon_max = float(lonModel_360.min()), float(lonModel_360.max())
    
        # Snap model domain to nearest ERA5 grid points
        lat_bounds = ERA5Data.sel(latitude=[lat_min, lat_max], method="nearest").latitude.values
        lon_bounds = ERA5Data.sel(longitude=[lon_min, lon_max], method="nearest").longitude.values
    
        # Ensure proper slicing order
        lat_start, lat_end = sorted(lat_bounds)[::-1]  # ERA5 latitude is descending
        lon_start, lon_end = sorted(lon_bounds)
    
        # Subset the ERA5 data
        ERA5_subset = ERA5Data.sel(
            latitude=slice(lat_start, lat_end),
            longitude=slice(lon_start, lon_end)
        )
    
        # Convert ERA5 longitudes from 0–360 → -180–180 (in-place update)
        ERA5_subset = ERA5_subset.assign_coords(
            longitude=(((ERA5_subset.longitude + 180) % 360) - 180)
        )
    
        # Sort longitudes to keep increasing order (optional, but helpful for plotting)
        ERA5_subset = ERA5_subset.sortby('longitude')
    
        return ERA5_subset

    @staticmethod
    def LoadERA5Data(DirectoryManager, ModelData, variableName, dataType = "Surface"):
        if dataType == "Surface":
            ERA5FileName = ERA5DataLoading_Class.GetERA5FileName_Surface(ERA5FilePath, variableName)
            print(ERA5FileName)
        elif dataType == "PressureLevels":
            ERA5FileName = ERA5DataLoading_Class.GetERA5FileName_PressureLevels(ERA5FilePath, variableName)
        
        
        ERA5_NAME_MAP = {
            "total_precipitation": "tp",
            "2m_temperature": "t2m",
            "surface_sensible_heat_flux": "sshf",
            "surface_latent_heat_flux": "slhf",
            "msl": "msl",  # already matching
        }
        ERA5variableName = ERA5_NAME_MAP.get(variableName, variableName)
        
        ERA5Data = xr.open_dataset(ERA5FileName)[ERA5variableName]
        ERA5_subset = ERA5DataLoading_Class.SubsetERA5(ERA5Data,ModelData)
        return ERA5_subset
    
    @staticmethod
    def SelectNearestERA5Time(ERA5Data, timeString):
        """
        Selects the nearest ERA5 time slice based on a custom time string.
        """
    
        # Convert custom timeString to pandas Timestamp
        time_dt = pd.to_datetime(timeString.replace('_', ' ').replace('.', ':'))
    
        # Use xarray's nearest selection
        ERA5Data_t = ERA5Data.sel(valid_time=time_dt, method='nearest')
    
        return ERA5Data_t